In [ ]:
# Import configuration
import json
from datetime import datetime

def load_config(config_path="config.json"):
    """Load configuration from a JSON file."""
    with open(config_path, 'r') as file:
        config = json.load(file)
    return config["sqlserver_name"], config["sqlserver_db"], config["sqlserver_ip"], config["sqlserver_port"], config["sqlserver_user"], config["sqlserver_pwd"], config['base_url'], config['username'], config['password'], config['output_directory'], config['pd_directory']
    

# Test loading configuration
sqlserver_name, sqlserver_db, sqlserver_ip, sqlserver_port, sqlserver_user, sqlserver_pwd, base_url, username, password, output_dir, pd_directory = load_config()
print("Configuration loaded successfully.")

In [ ]:
# Establish a database server connection
import pyodbc
from sqlalchemy import create_engine

def odbc_escape(value: str) -> str:
    # Wrap in braces and double any closing brace to escape it
    return "{" + value.replace("}", "}}") + "}"

server = f"{sqlserver_ip},{sqlserver_port}"
conn_str = (
    "Driver={ODBC Driver 17 for SQL Server};"
    f"Server={odbc_escape(server)};"
    f"Database={odbc_escape(sqlserver_db)};"
    f"UID={odbc_escape(sqlserver_user)};"
    f"PWD={odbc_escape(sqlserver_pwd)};"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

sql_conn = pyodbc.connect(conn_str, autocommit=False)

cursor = sql_conn.cursor()
cursor.execute('SELECT schNo, schName FROM Schools')

for i, row in enumerate(cursor):
    if i >= 3:
        break
    print(row)

# 📌 Consume remaining results so connection is clean
while cursor.nextset():
    pass

cursor.close()

In [ ]:
# Find excel workbook to load
import os

# Find PD sample files to upload, skipping unwanted ones
def find_sample_files(directory, skip_prefix="PD-source-data-workbook", extension=".xlsx"):
    """Scan directory and list files NOT starting with skip_prefix, matching extension."""
    files = []
    for filename in os.listdir(directory):
        if filename.endswith(extension) and not filename.startswith(skip_prefix):
            files.append(os.path.join(directory, filename))
    return files

# Load sample files
sample_files = find_sample_files(pd_directory)

#print(f"Found {len(sample_files)} sample files to upload:")
#for f in sample_files:
#    print("-", os.path.basename(f))

# Load all excel workbook in memory.
import openpyxl

# Load all Excel files into memory (workbooks)
def load_excel_workbooks(file_list):
    """Load Excel workbooks into memory."""
    workbooks = {}
    for file_path in file_list:
        try:
            wb = openpyxl.load_workbook(file_path)
            workbooks[file_path] = wb
            print(f"✅ Loaded: {os.path.basename(file_path)}")
        except Exception as e:
            print(f"❌ Failed to load {os.path.basename(file_path)}: {e}")
    return workbooks

# Actually load them now
all_workbooks = load_excel_workbooks(sample_files)

print(f"\nTotal workbooks loaded: {len(all_workbooks)}")

In [ ]:
# Convert Excel workbook data into XML
import xml.etree.ElementTree as ET

def workbook_to_xml(wb):
    ws = wb["PD data"]  # your sheet is called exactly "PD data"

    # Build a list of column headers
    headers = []
    for cell in ws[1]:
        if cell.value is not None:
            headers.append(cell.value.strip())
        else:
            headers.append("")

    # Column mapping (your provided one)
    column_mapping = {
        "PD Name": "PDName",
        "PD Format": "PDFormat",
        "PD Focus": "PDFocus",
        "Location": "Location",
        "Year": "Year",
        "Start Date (YYYY-MM-DD)": "StartDate",
        "End Date (YYYY-MM-DD)": "EndDate",
        "Duration in Days": "DurationDays",
        "Duration in Hours": "DurationHours",
        "Teacher PF Number": "TeacherPFNumber",
        "Teacher First Name": "TeacherFirstName",
        "Teacher Last Name": "TeacherLastName",
        "Gender": "Gender",
        "Disability": "Disability",
        "Approximate Years Teaching": "ApproximateYearsTeaching",
        "80% Attendance": "Percent80Attendance",
        "Statement of Completion": "StatementCompletion",
        "School": "School",
        "Total Teachers in School": "TotalTeachersInSchool",
        "Teachers Attending from School": "TeachersAttendingFromSchool",
        "Attendance Rate": "AttendanceRate"

    }
    # Add attended day columns dynamically
    for i in range(1, 16):
        column_mapping[f"Attended Day {i}"] = f"AttendedDay{i}"

    # Root element
    root = ET.Element("ListObject")
    root.set("FirstRow", "2")

    # Set pdName and pdYear attributes
    first_data_row = list(ws.iter_rows(min_row=2, max_row=2, values_only=True))[0]
    header_to_index = {h: i for i, h in enumerate(headers)}
    root.set("pdName", str(first_data_row[header_to_index["PD Name"]]))
    start_date = first_data_row[header_to_index["Start Date (YYYY-MM-DD)"]]
    root.set("pdStartDate", start_date.strftime("%Y-%m-%d"))

    # Build rows
    for idx, row in enumerate(ws.iter_rows(min_row=2, values_only=True)):
        if all(cell is None for cell in row):
            continue  # Skip blank rows
        
        row_elem = ET.SubElement(root, "row")
        row_elem.set("Index", str(idx))
        for header, cell_value in zip(headers, row):
            if not header:
                continue  # Skip empty headers
            
            xml_attr = column_mapping.get(header)
            if xml_attr:
                if header in ["Start Date (YYYY-MM-DD)", "End Date (YYYY-MM-DD)"]:
                    if cell_value is not None:
                        # Convert dates to Excel serial number
                        val = (cell_value - datetime(1899, 12, 30)).days
                    else:
                        val = ""
                else:
                    val = cell_value if cell_value is not None else ""
                
                row_elem.set(xml_attr, str(val).strip())

    return ET.tostring(root, encoding="unicode")



In [ ]:
# Print out one generated XML, formatted nicely
somefile = next(iter(all_workbooks.keys()))
raw_xml = workbook_to_xml(all_workbooks[somefile])

# Insert newline before each <row Index=
formatted_xml = raw_xml.replace('<row Index=', '\n<row Index=')

print(formatted_xml)

In [ ]:
# Get all lookups from EMIS. This depends on notebook lookups.ipynb which needs to run at least once
# to pickle the data locally.
import os
import pickle

cache_dir = "cached-data"

with open(os.path.join(cache_dir, "lookups_data.pkl"), "rb") as f:
    lookups = pickle.load(f)

print(f"Loaded {len(lookups)} lookup from cache.")

In [ ]:
def extract_pd_metadata(wb):
    """Extract PD Name and PD Year from the workbook."""
    # Access the 'PD data' sheet
    ws = wb["PD data"]

    # Read the first data row (row 2)
    first_row = [cell.value for cell in ws[2]]

    # Read headers from row 1
    headers = [cell.value for cell in ws[1]]

    # Create dictionary of {column name -> value}
    row_dict = dict(zip(headers, first_row))

    # Extract PD Name and Year
    pd_name = row_dict.get("PD Name", "")
    pd_start_date = row_dict.get("Start Date (YYYY-MM-DD)", "")

    # Get the PD Code from lookups_data
    pd_code = next((item['C'] for item in lookups['teacherPdTypes'] if item['N'] == pd_name), None)

    if pd_code is None:
        raise ValueError(f"PD Name '{pd_name}' not found in lookup lkpTeacherPdTypes in workbook {wb.properties.title}.")
        
    if not pd_name or not pd_start_date:
        raise ValueError(f"Missing PD Name or Start Date in workbook {wb.properties.title}.")

    return pd_name, pd_code, pd_start_date

In [ ]:
# Code to process the Excel workbook into XML data ready for SQL server stored proc
import uuid
import pyodbc
import base64

def load_single_workbook_to_sql(file_path, sql_conn):
    wb = all_workbooks[file_path]
    xml_data = workbook_to_xml(wb)
    
    pd_name, pd_code, pd_start_date = extract_pd_metadata(wb)
    file_reference = str(uuid.uuid4())
    username = "ghachey@purltek.com"

    # Encode XML
    xml_base64 = base64.b64encode(xml_data.encode('utf-8')).decode('ascii')

    query = f"""
    DECLARE @p1 XML;
    DECLARE @bin VARBINARY(MAX);

    SET @bin = CAST(CAST('{xml_base64}' AS XML).value('.', 'VARBINARY(MAX)') AS VARBINARY(MAX));
    SET @p1 = CONVERT(XML, @bin);

    EXEC pTeacherWrite.LoadTeacherPd 
        @pdData = @p1, 
        @fileReference = '{file_reference}',
        @user = '{username}',
        @pdCode = '{pd_code}',
        @pdStartDate = '{pd_start_date}';
    """

    cursor = sql_conn.cursor()
    cursor.execute(query)

    # 📌 Important: advance through ALL possible result sets
    while cursor.nextset():
        pass

    sql_conn.commit()
    cursor.close()

    print(f"✅ Successfully loaded {os.path.basename(file_path)} into SQL Server.")

In [ ]:
# Try loading a single workbook in DB
somefile = list(all_workbooks.keys())[3] # change index to load load different one
print(f"Loading file: {os.path.basename(somefile)}")

# Run the loader
load_single_workbook_to_sql(somefile, sql_conn)

In [ ]:
# Bulk load all workbooks to SQL Server
import time

def bulk_load_all_workbooks(all_workbooks, sql_conn, verbose=True):
    total = len(all_workbooks)
    success_count = 0
    fail_count = 0
    start_time = time.time()

    for idx, file_path in enumerate(all_workbooks.keys(), start=1):
        try:
            print(f"({idx}/{total}) Loading: {os.path.basename(file_path)} ...", end=" ")
            load_single_workbook_to_sql(file_path, sql_conn)
            success_count += 1
            if verbose:
                print("✅")
        except Exception as e:
            fail_count += 1
            print(f"❌ Failed: {e}")

    duration = time.time() - start_time
    print("\n=== Bulk Load Summary ===")
    print(f"Total Files Attempted : {total}")
    print(f"✅ Success             : {success_count}")
    print(f"❌ Failed              : {fail_count}")
    print(f"⏱️ Duration           : {duration:.2f} seconds")

# Run it:
bulk_load_all_workbooks(all_workbooks, sql_conn)